# TinyML Lab (2 hours): Train a Small NN → Convert to TFLite → Deployment (Conceptual)

**Module:** IT4021 – Internet of Things and Big Data Analytics (Year 4)  
**Lab Focus:** TinyML workflow (training in Colab, inference on microcontroller)

## What you will do
1. Generate a small **sensor-like** dataset (synthetic, fast).
2. Train a tiny neural network in Keras (TensorFlow).
3. Convert the trained model to **TensorFlow Lite (TFLite)**.
4. *(Optional)* Quantize the model (smaller/faster for microcontrollers).
5. Export:
   - `model.tflite` (binary model file)
   - `model.h` (C header as a byte array for embedded firmware)
6. Understand deployment steps **conceptually** (TinyML pipeline).

> **Important:** Training happens on a PC/Colab. Microcontrollers typically run **inference** only.


## 0) Setup
Run this cell to install/import dependencies.

In [ ]:
# If TensorFlow is not available, install a CPU version (Colab usually already has TF).
# NOTE: Uncomment if needed.
# !pip -q install tensorflow==2.15.0

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os, pathlib, textwrap

print("TensorFlow version:", tf.__version__)
np.random.seed(42)
tf.random.set_seed(42)


## 1) Create a Tiny "Sensor" Dataset (Synthetic)

We simulate a single sensor reading (e.g., LDR / potentiometer / distance), normalized to **0–1**.

- **NORMAL**: values around a typical range with mild noise  
- **ABNORMAL**: values that deviate (but still with overlap/noise so it isn't trivial)

This keeps the lab fast and avoids data collection overhead.


In [ ]:
# Generate synthetic sensor-like dataset
N = 4000

# Normal readings: centered near 0.35 with noise
x_normal = np.clip(np.random.normal(loc=0.35, scale=0.08, size=(N//2, 1)), 0.0, 1.0)

# Abnormal readings: mixture of higher and lower values, with noise and overlap
x_ab1 = np.clip(np.random.normal(loc=0.75, scale=0.10, size=(N//4, 1)), 0.0, 1.0)
x_ab2 = np.clip(np.random.normal(loc=0.10, scale=0.08, size=(N//4, 1)), 0.0, 1.0)
x_abnormal = np.vstack([x_ab1, x_ab2])

X = np.vstack([x_normal, x_abnormal]).astype(np.float32)

y_normal = np.zeros((N//2,), dtype=np.int32)     # 0 = NORMAL
y_abnormal = np.ones((N//2,), dtype=np.int32)    # 1 = ABNORMAL
y = np.concatenate([y_normal, y_abnormal])

# Shuffle
idx = np.random.permutation(N)
X = X[idx]
y = y[idx]

# Train/val split
split = int(0.8 * N)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)
print("Class balance (train):", np.bincount(y_train), "val:", np.bincount(y_val))

# Visualize distribution
plt.figure()
plt.hist(X_train[y_train==0], bins=40, alpha=0.6, label="NORMAL (0)")
plt.hist(X_train[y_train==1], bins=40, alpha=0.6, label="ABNORMAL (1)")
plt.title("Synthetic Sensor Data Distribution (Normalized 0–1)")
plt.legend()
plt.show()


## 2) Build and Train a Tiny Neural Network

We train a tiny Dense model suitable for microcontrollers:

- Input: 1 feature  
- Hidden: 8 neurons  
- Output: 2 classes (softmax)


In [ ]:
# Define a tiny NN
model = keras.Sequential([
    layers.Input(shape=(1,), name="sensor_norm"),
    layers.Dense(8, activation="relu"),
    layers.Dense(2, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    verbose=1
)

# Plot training curves
plt.figure()
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.title("Accuracy")
plt.legend()
plt.show()

plt.figure()
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.title("Loss")
plt.legend()
plt.show()

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation accuracy: {val_acc:.3f}")


## 3) Quick Test: Predict on a Few Samples

In [ ]:
test_values = np.array([[0.12], [0.32], [0.42], [0.78], [0.92]], dtype=np.float32)
probs = model.predict(test_values)

for x, p in zip(test_values.flatten(), probs):
    pred = int(np.argmax(p))
    print(f"x={x:.2f} -> probs(NORMAL,ABNORMAL)={p}  pred={pred} ({'NORMAL' if pred==0 else 'ABNORMAL'})")


## 4) Convert Keras Model → TFLite

In [ ]:
# Convert to TFLite (float model)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

out_dir = pathlib.Path("export")
out_dir.mkdir(exist_ok=True)

tflite_path = out_dir / "model_float.tflite"
tflite_path.write_bytes(tflite_model)

print("Saved:", tflite_path, "size:", tflite_path.stat().st_size, "bytes")


## 5) (Optional) Quantize the Model (Dynamic Range)

Quantization reduces model size and often improves speed.  
This is a quick, lab-friendly quantization method.


In [ ]:
converter_q = tf.lite.TFLiteConverter.from_keras_model(model)
converter_q.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_q = converter_q.convert()

tflite_q_path = out_dir / "model_dynamic.tflite"
tflite_q_path.write_bytes(tflite_model_q)

print("Saved:", tflite_q_path, "size:", tflite_q_path.stat().st_size, "bytes")


## 6) Validate the TFLite Models (Inference in Python)

In [ ]:
def run_tflite_inference(tflite_bytes, inputs: np.ndarray):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    results = []
    for x in inputs.astype(np.float32):
        interpreter.set_tensor(input_details[0]["index"], x.reshape(1, 1))
        interpreter.invoke()
        out = interpreter.get_tensor(output_details[0]["index"]).reshape(-1)
        results.append(out)
    return np.array(results)

test_values = np.array([[0.12],[0.32],[0.42],[0.78],[0.92]], dtype=np.float32)

float_out = run_tflite_inference(tflite_model, test_values)
dyn_out = run_tflite_inference(tflite_model_q, test_values)

print("Float TFLite outputs:")
print(float_out)
print("\nDynamic quantized TFLite outputs:")
print(dyn_out)


## 7) Convert `model.tflite` → `model.h` (C Header)

Microcontroller firmware often includes the model as a C byte array (`model.h`) compiled into flash.


In [ ]:
def tflite_to_c_array(tflite_bytes: bytes, var_name: str) -> str:
    hex_array = ", ".join([f"0x{b:02x}" for b in tflite_bytes])
    return textwrap.dedent(f"""    #ifndef {var_name.upper()}_H
    #define {var_name.upper()}_H

    #include <cstdint>
    #include <cstddef>

    alignas(16) const unsigned char {var_name}[] = {{
      {hex_array}
    }};

    const unsigned int {var_name}_len = sizeof({var_name});

    #endif
    """).strip()

h_float = tflite_to_c_array(tflite_model, "model_float_tflite")
h_dyn = tflite_to_c_array(tflite_model_q, "model_dynamic_tflite")

float_h_path = out_dir / "model_float.h"
dyn_h_path = out_dir / "model_dynamic.h"

float_h_path.write_text(h_float)
dyn_h_path.write_text(h_dyn)

print("Generated headers:")
print("-", float_h_path, "bytes:", float_h_path.stat().st_size)
print("-", dyn_h_path, "bytes:", dyn_h_path.stat().st_size)


## 8) Deployment on ESP32 (Conceptual)

### Artifacts you take from Colab to Arduino
- `model_float.h` *(or `model_dynamic.h`)*

### Embedded inference steps
1. Include the model header.
2. Create a **tensor arena** (static byte buffer).
3. Create the TFLite Micro interpreter (model + ops + arena).
4. `AllocateTensors()`.
5. Read sensor → normalize exactly like training (e.g., `raw/4095.0`).
6. Set input tensor values.
7. `Invoke()`.
8. Read output probabilities → trigger action (LED / MQTT / etc.).

### TinyML pipeline
`Sensor → Normalize/Features → TFLite Micro → Prediction → Action`

> If normalization is different at deployment, predictions will be unreliable.


## 9) Download Exported Files
Run this cell in Colab to download outputs.

In [ ]:
try:
    from google.colab import files
    for p in [
        "export/model_float.tflite",
        "export/model_dynamic.tflite",
        "export/model_float.h",
        "export/model_dynamic.h",
    ]:
        if os.path.exists(p):
            files.download(p)
except Exception as e:
    print("If you're not in Colab, open the export/ folder locally.")
    print("Error:", e)
